In [1]:
import json
from pathlib import Path

import pandas as pd

# Data preparation and exploration

In [2]:
# Load the data
results_path = Path("results/results.csv")
df = pd.read_csv(results_path)
df.head(10)

,0.025,0.05,0.25,0.75,0.95,0.975,estimate,stat,method,data_seed,boot_seed,n_l3,n_l2,n_l1,rand_eff_dgp,replication_id
0,-0.337382,-0.283545,-0.132109,0.065388,0.216824,0.270660,-0.033361,mu,profile-likelihood,8047133337562467477,277495415,15,4,8,norm,0
1,0.295696,0.324712,0.418288,0.579573,0.737470,0.800084,0.492631,sd_l3,profile-likelihood,8047133337562467477,277495415,15,4,8,norm,0
2,0.388533,0.403980,0.455750,0.540944,0.614754,0.641528,0.496151,sd_l2,profile-likelihood,8047133337562467477,277495415,15,4,8,norm,0
3,-0.313842,-0.259507,-0.129249,0.062600,0.215148,0.267657,-0.033361,mu,parametric-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
4,0.186325,0.249476,0.369539,0.527449,0.657351,0.699028,0.492631,sd_l3,parametric-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
5,0.367022,0.386253,0.454266,0.534494,0.595825,0.617124,0.496151,sd_l2,parametric-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
6,-0.364098,-0.300817,-0.136757,0.072522,0.215303,0.283126,-0.033361,mu,cases-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
7,0.253412,0.304335,0.439163,0.604923,0.714001,0.756105,0.492631,sd_l3,cases-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
8,0.344909,0.361420,0.423962,0.510710,0.567030,0.591456,0.496151,sd_l2,cases-percentile-boot,8047133337562467477,277495415,15,4,8,norm,0
9,-0.364098,-0.300817,-0.136757,0.072522,0.283126,0.425157,-0.033361,mu,cases-double-boot,8047133337562467477,277495415,15,4,8,norm,0


In [3]:
# df should have length 3 (dgp) * 3 (functionals) * 2 (sizes) * 4 (methods) * 500 (repetitions) = 36000
len(df)

36000

In [4]:
# Profile likelihood failed to produde a CI fir the third level standard
# deviation in 4 cases.
df[df.isna().any(axis=1)]

,0.025,0.05,0.25,0.75,0.95,0.975,estimate,stat,method,data_seed,boot_seed,n_l3,n_l2,n_l1,rand_eff_dgp,replication_id
13093,NaN,NaN,NaN,NaN,NaN,NaN,0.000113,sd_l3,profile-likelihood,11895652192426520676,42006704,15,4,8,t,91
24529,NaN,NaN,NaN,NaN,NaN,NaN,0.000073,sd_l3,profile-likelihood,618583818986726600,287258293,15,4,8,lognorm,44
26533,NaN,NaN,NaN,NaN,NaN,NaN,0.000008,sd_l3,profile-likelihood,15047725419429258025,1905345484,15,4,8,lognorm,211
33841,NaN,NaN,NaN,NaN,NaN,NaN,0.516234,sd_l3,profile-likelihood,4077804771184112143,1650368957,50,4,8,lognorm,320


In [5]:
# Drop intervals with unavailable endpoints from the coverage analysis.
# Four profile-likelihood intervals are unavailable in these results.
ci_columns = ["0.025", "0.05", "0.25", "0.75", "0.95", "0.975"]
df = df.dropna(subset=ci_columns).copy().reset_index(drop=True)
len(df)

35996

In [6]:
import numpy as np

# Read the config to get the actual GT values
config_path = Path("results/config.json")
with open(config_path) as f:
    config = json.load(f)

gt_params = {
    "mu": config["mu"],
    "sd_l3": np.sqrt(config["var_l3"]),
    "sd_l2": np.sqrt(config["var_l2"]),
}

In [7]:
# Coverage indicators for two sided intervals
ci_bounds = {
    50: ("0.25", "0.75"),
    90: ("0.05", "0.95"),
    95: ("0.025", "0.975"),
}
true_value = df["stat"].map(lambda e: gt_params[e])

for level, (lower, upper) in ci_bounds.items():
    df[f"{level}_covers"] = (
        df[lower].le(true_value) & true_value.le(df[upper])
    ).astype(int)

df[["stat", "method", *[f"{level}_covers" for level in ci_bounds]]].head()

,stat,method,50_covers,90_covers,95_covers
0,mu,profile-likelihood,1,1,1
1,sd_l3,profile-likelihood,1,1,1
2,sd_l2,profile-likelihood,0,1,1
3,mu,parametric-percentile-boot,1,1,1
4,sd_l3,parametric-percentile-boot,0,1,1


In [11]:
# Compute the empirical coverage across different configurations
# Only the level 3 number of groups change
group_cols = [
    "rand_eff_dgp",
    "stat",
    "n_l3",
    "method",
]

coverage_summary = (
    df.groupby(group_cols)
    .agg(
        coverage_50_estimate=("50_covers", "mean"),
        coverage_90_estimate=("90_covers", "mean"),
        coverage_95_estimate=("95_covers", "mean"),
        n_repetitions=("replication_id", "count"),
    )
    .reset_index()
)

In [12]:
coverage_summary

,rand_eff_dgp,stat,n_l3,method,coverage_50_estimate,coverage_90_estimate,coverage_95_estimate,n_repetitions
0,lognorm,mu,15,cases-double-boot,0.514000,0.894000,0.968000,500
1,lognorm,mu,15,cases-percentile-boot,0.514000,0.878000,0.932000,500
2,lognorm,mu,15,parametric-percentile-boot,0.434000,0.832000,0.872000,500
3,lognorm,mu,15,profile-likelihood,0.452000,0.840000,0.894000,500
4,lognorm,mu,50,cases-double-boot,0.444000,0.884000,0.964000,500
...,...,...,...,...,...,...,...,...
67,t,sd_l3,15,profile-likelihood,0.288577,0.641283,0.727455,499
68,t,sd_l3,50,cases-double-boot,0.196000,0.476000,0.552000,500
69,t,sd_l3,50,cases-percentile-boot,0.506000,0.886000,0.930000,500
70,t,sd_l3,50,parametric-percentile-boot,0.192000,0.466000,0.536000,500


In [16]:
# Compute BCa CIs for the coverage
from scipy.stats import bootstrap

results = []

for group_values, group_df in df.groupby(group_cols):
    row = {col: val for col, val in zip(group_cols, group_values)}

    for level in ci_bounds:
        coverage_indicators = group_df[f"{level}_covers"]

        coverage_ci = bootstrap(
            (coverage_indicators,),
            np.mean,
            n_resamples=10_000,
            confidence_level=0.95,
            method="BCa",
            rng=np.random.default_rng(0),
        )

        low, high = coverage_ci.confidence_interval

        row[f"coverage_{level}_estimate"] = coverage_indicators.mean()
        row[f"coverage_{level}_ci_low"] = low
        row[f"coverage_{level}_ci_high"] = high

    results.append(row)

coverage_ci_df = pd.DataFrame(results)

In [17]:
coverage_ci_df

,rand_eff_dgp,stat,n_l3,method,coverage_50_estimate,coverage_50_ci_low,coverage_50_ci_high,coverage_90_estimate,coverage_90_ci_low,coverage_90_ci_high,coverage_95_estimate,coverage_95_ci_low,coverage_95_ci_high
0,lognorm,mu,15,cases-double-boot,0.514000,0.470000,0.558000,0.894000,0.864000,0.920000,0.968000,0.950000,0.982000
1,lognorm,mu,15,cases-percentile-boot,0.514000,0.470000,0.558000,0.878000,0.848000,0.904000,0.932000,0.908000,0.952000
2,lognorm,mu,15,parametric-percentile-boot,0.434000,0.392000,0.478000,0.832000,0.798000,0.864000,0.872000,0.840000,0.900000
3,lognorm,mu,15,profile-likelihood,0.452000,0.408000,0.496000,0.840000,0.806000,0.870000,0.894000,0.864000,0.920000
4,lognorm,mu,50,cases-double-boot,0.444000,0.402000,0.488000,0.884000,0.854000,0.910000,0.964000,0.944000,0.978000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,t,sd_l3,15,profile-likelihood,0.288577,0.250501,0.330661,0.641283,0.599198,0.683367,0.727455,0.687375,0.765531
68,t,sd_l3,50,cases-double-boot,0.196000,0.164000,0.234000,0.476000,0.434000,0.520000,0.552000,0.508000,0.594000
69,t,sd_l3,50,cases-percentile-boot,0.506000,0.462000,0.548000,0.886000,0.856000,0.912000,0.930000,0.904000,0.950000
70,t,sd_l3,50,parametric-percentile-boot,0.192000,0.160000,0.228000,0.466000,0.422000,0.510000,0.536000,0.492000,0.580000


In [19]:
# Store the results
result_df_path = Path("results/analysis_tables")
result_df_path.mkdir(exist_ok=True, parents=True)
coverage_ci_df.to_csv(result_df_path / "results_coverage_ci.csv", index=False)